In [1]:
import pandas as pd
import numpy as np

customers = pd.DataFrame({
    "CustomerID": [101, 102, 103, 104, 105, 106],
    "CustomerName": [
        "Ananya", "Rahul", "Meera",
        "Arjun", "Priya", "Kiran"
    ],
    "City": [
        "New York", "Boston", "Chicago",
        "Boston", "New York", "Seattle"
    ],
    "CustomerSegment": [
        "Premium", "Regular", "Premium",
        "Regular", "Premium", "Regular"
    ]
})

orders = pd.DataFrame({
    "OrderID": [1001, 1002, 1003, 1004, 1005, 1006, 1007],
    "CustomerID": [101, 102, 101, 104, 105, 102, 107],
    "Product": [
        "Laptop", "Phone", "Tablet", "Laptop",
        "Phone", "Laptop", "Tablet"
    ],
    "Amount": [1200, 800, 400, 1500, 750, 1100, 350]
})

payments = pd.DataFrame({
    "OrderID": [1001, 1002, 1003, 1005, 1006, 1008],
    "PaymentMethod": [
        "Card", "PayPal", "Card",
        "Cash", "Card", "PayPal"
    ],
    "PaymentStatus": [
        "Paid", "Paid", "Pending",
        "Paid", "Paid", "Paid"
    ]
})

In [2]:
#Perform an inner join between customers and orders using CustomerID
customer_orders = pd.merge(
    customers,
    orders,
    on="CustomerID",
    how="inner"
)

customer_orders

,CustomerID,CustomerName,City,CustomerSegment,OrderID,Product,Amount
0,101,Ananya,New York,Premium,1001,Laptop,1200
1,101,Ananya,New York,Premium,1003,Tablet,400
2,104,Arjun,Boston,Regular,1004,Laptop,1500
3,102,Rahul,Boston,Regular,1002,Phone,800
4,102,Rahul,Boston,Regular,1006,Laptop,1100
5,105,Priya,New York,Premium,1005,Phone,750


In [3]:
#Perform a left join to keep every customer, including those without orders.
left_join = pd.merge(
    customers,
    orders,
    on="CustomerID",
    how="left"
)

left_join

,CustomerID,CustomerName,City,CustomerSegment,OrderID,Product,Amount
0,101,Ananya,New York,Premium,1001.0,Laptop,1200.0
1,101,Ananya,New York,Premium,1003.0,Tablet,400.0
2,102,Rahul,Boston,Regular,1002.0,Phone,800.0
3,102,Rahul,Boston,Regular,1006.0,Laptop,1100.0
4,103,Meera,Chicago,Premium,NaN,NaN,NaN
5,104,Arjun,Boston,Regular,1004.0,Laptop,1500.0
6,105,Priya,New York,Premium,1005.0,Phone,750.0
7,106,Kiran,Seattle,Regular,NaN,NaN,NaN


In [4]:
#Perform a right join to keep every order, including orders from unknown customers.
right_join = pd.merge(
    customers,
    orders,
    on="CustomerID",
    how="right"
)

right_join

,CustomerID,CustomerName,City,CustomerSegment,OrderID,Product,Amount
0,101,Ananya,New York,Premium,1001,Laptop,1200
1,102,Rahul,Boston,Regular,1002,Phone,800
2,101,Ananya,New York,Premium,1003,Tablet,400
3,104,Arjun,Boston,Regular,1004,Laptop,1500
4,105,Priya,New York,Premium,1005,Phone,750
5,102,Rahul,Boston,Regular,1006,Laptop,1100
6,107,NaN,NaN,NaN,1007,Tablet,350


In [7]:
#Perform an outer join to identify customers and orders that do not match.
outer_join = pd.merge(
    customers,
    orders,
    on="CustomerID",
    how="outer",
    indicator=True
)

outer_join


#To show only unmatched records:
#outer_join[outer_join["_merge"] != "both"]

,CustomerID,CustomerName,City,CustomerSegment,OrderID,Product,Amount,_merge
0,101,Ananya,New York,Premium,1001.0,Laptop,1200.0,both
1,101,Ananya,New York,Premium,1003.0,Tablet,400.0,both
2,102,Rahul,Boston,Regular,1002.0,Phone,800.0,both
3,102,Rahul,Boston,Regular,1006.0,Laptop,1100.0,both
4,103,Meera,Chicago,Premium,NaN,NaN,NaN,left_only
5,104,Arjun,Boston,Regular,1004.0,Laptop,1500.0,both
6,105,Priya,New York,Premium,1005.0,Phone,750.0,both
7,106,Kiran,Seattle,Regular,NaN,NaN,NaN,left_only
8,107,NaN,NaN,NaN,1007.0,Tablet,350.0,right_only


In [8]:
#Merge orders with payments using OrderID.
order_payments = pd.merge(
    orders,
    payments,
    on="OrderID",
    how="left",
    indicator=True
)

order_payments

,OrderID,CustomerID,Product,Amount,PaymentMethod,PaymentStatus,_merge
0,1001,101,Laptop,1200,Card,Paid,both
1,1002,102,Phone,800,PayPal,Paid,both
2,1003,101,Tablet,400,Card,Pending,both
3,1004,104,Laptop,1500,NaN,NaN,left_only
4,1005,105,Phone,750,Cash,Paid,both
5,1006,102,Laptop,1100,Card,Paid,both
6,1007,107,Tablet,350,NaN,NaN,left_only


In [ ]:
#Identify orders with no matching payment record.
unpaid_records = order_payments[
    order_payments["_merge"] == "left_only"
]

unpaid_records[["OrderID", "CustomerID", "Product", "Amount"]]

#These orders have no matching record in the payments table. 
# That does not necessarily mean payment was never made, only that no payment record exists here.

,OrderID,CustomerID,Product,Amount
3,1004,104,Laptop,1500
6,1007,107,Tablet,350


In [10]:
#Identify customers who have never placed an order.
customers_without_orders = customers[
    ~customers["CustomerID"].isin(orders["CustomerID"])
]

customers_without_orders

,CustomerID,CustomerName,City,CustomerSegment
2,103,Meera,Chicago,Premium
5,106,Kiran,Seattle,Regular


In [11]:
#Calculate total order amount per customer, including customers with zero orders.
customer_totals = pd.merge(
    customers,
    orders.groupby("CustomerID", as_index=False)["Amount"].sum()
        .rename(columns={"Amount": "TotalOrderAmount"}),
    on="CustomerID",
    how="left"
)

customer_totals["TotalOrderAmount"] = (
    customer_totals["TotalOrderAmount"].fillna(0)
)

customer_totals

,CustomerID,CustomerName,City,CustomerSegment,TotalOrderAmount
0,101,Ananya,New York,Premium,1600.0
1,102,Rahul,Boston,Regular,1900.0
2,103,Meera,Chicago,Premium,0.0
3,104,Arjun,Boston,Regular,1500.0
4,105,Priya,New York,Premium,750.0
5,106,Kiran,Seattle,Regular,0.0


In [12]:
#Calculate total paid order amount per city. Count only orders whose PaymentStatus is "Paid".
paid_orders = pd.merge(
    orders,
    payments,
    on="OrderID",
    how="inner"
)

paid_orders = paid_orders[
    paid_orders["PaymentStatus"] == "Paid"
]

paid_customer_orders = pd.merge(
    paid_orders,
    customers[["CustomerID", "City"]],
    on="CustomerID",
    how="inner"
)

paid_by_city = paid_customer_orders.groupby("City")["Amount"].sum()

paid_by_city

City
Boston      1900
New York    1950
Name: Amount, dtype: int64

In [13]:
#Create a final customer summary containing customer name, city, number of orders, and total order amount.
order_summary = orders.groupby("CustomerID").agg(
    NumberOfOrders=("OrderID", "count"),
    TotalOrderAmount=("Amount", "sum")
).reset_index()

customer_summary = pd.merge(
    customers[["CustomerID", "CustomerName", "City"]],
    order_summary,
    on="CustomerID",
    how="left"
)

customer_summary[["NumberOfOrders", "TotalOrderAmount"]] = (
    customer_summary[["NumberOfOrders", "TotalOrderAmount"]]
    .fillna(0)
    .astype(int)
)

customer_summary

,CustomerID,CustomerName,City,NumberOfOrders,TotalOrderAmount
0,101,Ananya,New York,2,1600
1,102,Rahul,Boston,2,1900
2,103,Meera,Chicago,0,0
3,104,Arjun,Boston,1,1500
4,105,Priya,New York,1,750
5,106,Kiran,Seattle,0,0


Key takeaway: merge() combines tables, groupby() summarizes data, and isin() helps find matching or missing keys. Together, they cover a large share of everyday data preparation tasks.